## Step 4: Teaching a Robot to Spot Planets (CNN Training Pipeline)

The grand finale — after humans did the tedious labeling in Notebook 3, we hand the job over to a neural network so it can (hopefully) do it faster, more consistently, and without needing coffee breaks. This notebook takes the labeled CSVs, turns each FITS file into an image, and trains a CNN classifier to predict "transit" vs. "no transit."

**The Algorithm:**

1. **Load labeled file lists** via `read_filenames()`, a small utility that transparently handles `.csv`, `.xls`, and `.xlsx` inputs — reading from `Negative_File.csv`, `Positive_File.csv`, and treating `Skipped_file.csv` as the "uncertain" pool. It prints counts of each so you know what you're working with before committing to anything.

2. **Ask the user how to build the training set**, with two branches:
   - **Combine positives + uncertains** — the user picks how many of each to include (capped at what's available), and labels them all `1` (positive class) alongside negatives labeled `0`.
   - **Negative-only path** — comes with a stern warning that a single-class dataset is scientifically useless (about as useful as a compass that only points "down"), and gently nudges the user to add at least some positive samples before proceeding. If, after all input, only one unique label exists, the script exits early rather than wasting compute on a doomed training run.

3. **Load and preprocess each FITS file into a fixed-size image** via `load_fits_image()`:
   - Opens the file and scans through HDUs for the first one containing actual data.
   - If it's a structured/table HDU, it hunts for a usable flux column in priority order: `PDCSAP_FLUX` → `SAP_FLUX` → `FLUX` → `flux` → *any numeric column it can find* — this function really doesn't like taking "no" for an answer.
   - Flattens whatever array it finds (handling 1D, 2D, and multi-frame 3D data by taking the first frame) into a 1D sequence of values.
   - **Pads or truncates** that sequence to exactly `256 × 256 = 65,536` values, then reshapes it into a **256×256** 2D "image" — effectively turning a time-series/flux array into a picture a CNN can consume, even if the physical meaning of "pixel adjacency" here is a bit more abstract than a real photo.
   - `preprocess_image()` then min-max normalizes every image to the `[0, 1]` range (guarding against divide-by-zero for flat/constant images).

4. **Save the exact selection made** (`training_data_selection.pkl`, via `pickle`) — capturing which filenames and labels were chosen, so future-you (or a teammate) can reproduce this exact training run without re-answering all the input prompts.

5. **Let the user pick a model architecture:**
   - **Option 1 — Deep CNN** (`create_deeper_cnn`): 5 convolutional blocks with increasing filter counts (64 → 128 → 256 → 512 → 512), each paired with `BatchNormalization` (for training stability) and `MaxPooling2D` (for downsampling), followed by `GlobalAveragePooling2D` and two Dense layers (512 → 256 units) with Dropout (0.5, then 0.3) as regularization insurance against overfitting on a likely-small dataset. Output layer adapts automatically: sigmoid + binary cross-entropy for 2 classes, or softmax + sparse categorical cross-entropy for more.
   - **Option 2 — "CNN + LSTM"** (`create_cnn_lstm`): plot twist, it just prints a warning that there's no genuinely sequential (multi-frame-per-sample) data available, and quietly hands you back the same Deep CNN. 🎭 *(The LSTM was more of an aspiration than an implementation — a nice reminder that architecture names in code comments don't always match what actually runs.)*

6. **K-Fold Cross-Validation** using `sklearn.model_selection.KFold` with up to 6 folds (`k = min(6, n_samples)`), falling back to a simple 80/20 random split if there's only enough data for a single fold. This is done because judging your model on a single train/test split is like judging a restaurant from one bite — cross-validation gives a more honest estimate of how well the model generalizes.

7. **Train with data augmentation** via `ImageDataGenerator` — applying random rotations (±10°), width/height shifts (10%), zoom (10%), and horizontal flips, with `fill_mode='nearest'` for the edges created by these transforms. This artificially expands the effective diversity of a likely small dataset, helping the model generalize instead of memorizing exact pixel positions. Training runs for **40 epochs per fold** with a small `batch_size=4` (sensible given limited data), and a `ModelCheckpoint` callback saves the best-performing weights per fold based on validation accuracy.

8. **Track and persist the best model across all folds** — after each fold finishes, its best validation accuracy is compared against the running best; if it wins, that fold's model is saved as `best_2D-trained_model.keras`, overwriting the previous champion.

9. **Persist the fully processed dataset** as `2d_X.npy` and `2d_y.npy` — so the exact preprocessed image tensors and labels used for training can be reloaded later without re-running all the FITS-to-image conversion (your CPU/patience sends its thanks).

**Output:** A trained Keras model (`best_2D-trained_model.keras`), per-fold checkpoint models (`best_model_fold{N}.keras`), the training data selection record (`training_data_selection.pkl`), and the final processed tensors (`2d_X.npy`, `2d_y.npy`) — everything needed to reproduce, evaluate, or fine-tune the classifier later.

*In short: raw pixel time-series in, "is this a planet?" verdict out — with the neural network doing the astronomy equivalent of "spot the difference."*

In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
from astropy.io import fits
from sklearn.model_selection import KFold
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
def read_filenames(file_path, column='filename'):
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ('.xls', '.xlsx'):
        df = pd.read_excel(file_path)
    elif ext == '.csv':
        df = pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported file format: {ext}. Use .xls, .xlsx or .csv.")
    return df[column].dropna().tolist()

In [3]:
def load_fits_image(filepath, target_size=(64, 64)):
    with fits.open(filepath) as hdul:
        data = None
        for hdu in hdul:
            if hdu.data is not None:
                data = hdu.data
                break
        if data is None:
            raise ValueError(f"No data found in {filepath}")
        if data.dtype.names is not None:
            col_names = data.dtype.names
            flux_col = None
            for name in ['PDCSAP_FLUX', 'SAP_FLUX', 'FLUX', 'flux']:
                if name in col_names:
                    flux_col = name
                    break
            if flux_col is None:
                for name in col_names:
                    if np.issubdtype(data[name].dtype, np.number):
                        flux_col = name
                        break
            if flux_col is None:
                raise ValueError(f"No numeric column found in table {filepath}")
            arr = data[flux_col].flatten()
        else:
            arr = data
            if arr.ndim > 2:
                arr = arr[0, ...]
            if arr.ndim == 2:
                arr = arr.flatten()
            elif arr.ndim != 1:
                raise ValueError(f"Unsupported array dimension: {arr.ndim}")
        N = target_size[0] * target_size[1]
        if len(arr) < N:
            padded = np.zeros(N, dtype=np.float32)
            padded[:len(arr)] = arr[:N]
        else:
            padded = arr[:N]
        return padded.reshape(target_size)

In [4]:
def preprocess_image(img):
    img = img.astype(np.float32)
    min_val = img.min()
    max_val = img.max()
    if max_val - min_val > 1e-8:
        return (img - min_val) / (max_val - min_val)
    return np.zeros_like(img)

In [5]:
def create_deeper_cnn(input_shape, n_classes):
    model = models.Sequential([
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(n_classes if n_classes > 1 else 1,
                     activation='softmax' if n_classes > 1 else 'sigmoid')
    ])
    loss = 'sparse_categorical_crossentropy' if n_classes > 1 else 'binary_crossentropy'
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model

In [6]:
def create_cnn_lstm(input_shape, n_classes):
    print("\nWarning: CNN+LSTM requires sequential data (multiple images per sample).")
    print("Reverting to a 2D CNN.\n")
    return create_deeper_cnn(input_shape, n_classes)

In [7]:
def main():
    neg_files = read_filenames('Negative_File.csv')
    pos_files = read_filenames('Positive_File.csv')
    unc_files = read_filenames('Skipped_file.csv')

    print("File counts")
    print(f"Negative  : {len(neg_files)}")
    print(f"Positive  : {len(pos_files)}")
    print(f"Uncertain : {len(unc_files)}")
    print("\n")

    use_both = input(
        f"The number of positive files is {len(pos_files)}\n"
        f"The number of uncertain files is {len(unc_files)}. \nDo you wish to use them both? (y/n): "
    ).strip().lower()

    selected_files = []
    labels = []

    if use_both in ('y', 'yes'):
        while True:
            try:
                n_pos = int(input(f"How many positive files to use? (max {len(pos_files)}): "))
                if 0 <= n_pos <= len(pos_files):
                    break
                print(f"Please enter a number between 0 and {len(pos_files)}")
            except ValueError:
                print("Invalid input. Enter an integer.")
        while True:
            try:
                n_unc = int(input(f"How many uncertain files to use? (max {len(unc_files)}): "))
                if 0 <= n_unc <= len(unc_files):
                    break
                print(f"Please enter a number between 0 and {len(unc_files)}")
            except ValueError:
                print("Invalid input. Enter an integer.")

        pos_sample = pos_files[:n_pos] if n_pos > 0 else []
        unc_sample = unc_files[:n_unc] if n_unc > 0 else []
        selected_files = pos_sample + unc_sample
        labels = [1] * n_pos + [0] * n_unc
        print(f"\nSelected {n_pos} positive and {n_unc} uncertain files. Total: {len(selected_files)}")
    else:
        print("\nWARNING: You are selecting only negative files. This will result in a single class.")
        print("For a meaningful classification, you need at least one positive sample.")
        add_pos = input("Do you want to include some positive files as well? (y/n): ").strip().lower()
        if add_pos in ('y', 'yes'):
            while True:
                try:
                    n_pos = int(input(f"How many positive files to add? (max {len(pos_files)}): "))
                    if 0 <= n_pos <= len(pos_files):
                        break
                    print(f"Please enter a number between 0 and {len(pos_files)}")
                except ValueError:
                    print("Invalid input. Enter an integer.")
            pos_sample = pos_files[:n_pos] if n_pos > 0 else []
        else:
            pos_sample = []

        while True:
            try:
                n_neg = int(input(f"The number of negative files is {len(neg_files)}. How many do you wish to use? (max {len(neg_files)}): "))
                if 0 <= n_neg <= len(neg_files):
                    break
                print(f"Please enter a number between 0 and {len(neg_files)}")
            except ValueError:
                print("Invalid input. Enter an integer.")

        neg_sample = neg_files[:n_neg] if n_neg > 0 else []
        selected_files = pos_sample + neg_sample
        labels = [1] * len(pos_sample) + [0] * n_neg
        print(f"\nSelected {len(pos_sample)} positive and {n_neg} negative files. Total: {len(selected_files)}")

    if len(set(labels)) < 2:
        print("\nERROR: Still only one class present. Exiting.")
        sys.exit(1)

    with open('training_data_selection.pkl', 'wb') as f:
        pickle.dump({'filenames': selected_files, 'labels': labels}, f)

    print('\nWhich one of the model do you prefer:')
    print('    1. CNN')
    print('    2. CNN + LSTM')
    while True:
        try:
            model_choice = int(input("Enter your choice: "))
            if model_choice in (1, 2):
                break
            print("Invalid input. Enter 1 or 2.")
        except ValueError:
            print("Invalid input. Enter an integer.")

    TARGET_SIZE = (256, 256)

    X, y = [], []
    print("\nLoading data...")
    for fname, label in zip(selected_files, labels):
        full_path = os.path.join('.', fname)
        if not os.path.exists(full_path):
            print(f"Warning: file '{full_path}' not found. Skipping.")
            continue
        try:
            img = load_fits_image(full_path, target_size=TARGET_SIZE)
            img = preprocess_image(img)
            X.append(img)
            y.append(label)
        except Exception as e:
            print(f"Error loading '{full_path}': {e}. Skipping.")

    if not X:
        print("No valid data loaded. Exiting.")
        sys.exit(1)

    X = np.array(X)
    y = np.array(y)
    if X.ndim == 3:
        X = X[..., np.newaxis]

    print(f"Loaded data shape: {X.shape}, labels shape: {y.shape}")
    print(f"Unique labels: {np.unique(y)}")

    num_classes = len(np.unique(y))
    if num_classes < 2:
        print("\nERROR: Only one class present after loading. Exiting.")
        sys.exit(1)

    n_samples = X.shape[0]
    k = min(6, n_samples)
    if k < 2:
        k = 2 if n_samples >= 2 else 1

    if k == 1:
        split = int(0.8 * n_samples)
        indices = np.random.permutation(n_samples)
        folds = [(indices[:split], indices[split:])]
    else:
        kf = KFold(n_splits=k, shuffle=True, random_state=42)
        folds = list(kf.split(X))

    best_val_acc = -1.0
    best_model = None

    print(f"\nStarting training with {len(folds)} fold(s), 40 epochs each.\n")

    datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    for fold_no, (train_idx, val_idx) in enumerate(folds, 1):
        print(f"=== Fold {fold_no}/{len(folds)} ===")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if model_choice == 1:
            model = create_deeper_cnn(X.shape[1:], num_classes)
        else:
            model = create_cnn_lstm(X.shape[1:], num_classes)

        checkpoint = keras.callbacks.ModelCheckpoint(
            f'best_model_fold{fold_no}.keras',
            monitor='val_accuracy',
            save_best_only=True,
            mode='max'
        )

        batch_size = 4

        train_gen = datagen.flow(X_train, y_train, batch_size=batch_size)
        val_gen = datagen.flow(X_val, y_val, batch_size=batch_size) 

        history = model.fit(
            train_gen,
            steps_per_epoch=len(X_train) // batch_size,
            epochs=40,
            validation_data=(X_val, y_val),
            callbacks=[checkpoint],
            verbose=1
        )

        val_acc = max(history.history['val_accuracy'])
        print(f"Fold {fold_no} best val accuracy: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model
            model.save('best_2D-trained_model.keras')

    print(f"\nBest validation accuracy across all folds: {best_val_acc:.4f}")
    print("Best model saved as 'best_2D-trained_model.keras'")

    np.save('2d_X.npy', X)
    np.save('2d_y.npy', y)
    print("Training data (X and y) saved as '2d_data_X.npy' and 'training_data_y.npy'.")

    print("\nAll done! Relax")

In [8]:
if __name__ == "__main__":
    main()

File counts
Negative  : 17
Positive  : 16
Uncertain : 1




The number of positive files is 16
The number of uncertain files is 1. 
Do you wish to use them both? (y/n):  n



For a meaningful classification, you need at least one positive sample.


Do you want to include some positive files as well? (y/n):  y
How many positive files to add? (max 16):  16
The number of negative files is 17. How many do you wish to use? (max 17):  16



Selected 16 positive and 16 negative files. Total: 32

Which one of the model do you prefer:
    1. CNN
    2. CNN + LSTM


...


Loading data...
Loaded data shape: (32, 256, 256, 1), labels shape: (32,)
Unique labels: [0 1]

Starting training with 6 fold(s), 40 epochs each.

=== Fold 1/6 ===

Reverting to a 2D CNN.

Epoch 1/40


C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 862ms/step - accuracy: 0.4545 - loss: 0.9365 - val_accuracy: 0.5000 - val_loss: 0.6947
Epoch 2/40
1/6 ━━━━━━━━━━━━━━━━━━━━ 3s 727ms/step - accuracy: 0.0000e+00 - loss: 0.6964

C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.0000e+00 - loss: 0.6964 - val_accuracy: 0.5000 - val_loss: 0.6939
Epoch 3/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 709ms/step - accuracy: 0.6818 - loss: 1.3608 - val_accuracy: 0.5000 - val_loss: 1.2330
Epoch 4/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.5000 - loss: 0.6991 - val_accuracy: 0.5000 - val_loss: 1.2796
Epoch 5/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 843ms/step - accuracy: 0.5909 - loss: 1.3603 - val_accuracy: 0.5000 - val_loss: 1.1040
Epoch 6/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.5000 - loss: 2.0102 - val_accuracy: 0.5000 - val_loss: 1.9611
Epoch 7/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 697ms/step - accuracy: 0.5000 - loss: 1.5338 - val_accuracy: 0.5000 - val_loss: 2.8253
Epoch 8/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.5000 - loss: 0.6998 - val_accuracy: 0.5000 - val_loss: 2.1479
Epoch 9/40
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 764ms/step - accuracy: 0.4091 - loss: 1.4869 - val_accuracy: 0.5000 - val_loss: 1.6795
Epo